# Backbone Experiment: convnext_base

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root
from config import Config
from data.cifake import get_cifake_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from tqdm.notebook import tqdm
import pandas as pd
import torch

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device: cuda
GPU:    NVIDIA RTX PRO 4000 Blackwell
VRAM:   25.2 GB


## Shared Config

In [2]:
# Only line to change
BACKBONE = "convnext_base"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.cifake_root    = "../data/raw/cifake"
    cfg.data.num_workers    = 4 # change to 0 if on Windows
    cfg.data.batch_size     = 64
    cfg.data.jpeg_aug = True
    cfg.backbone.name       = BACKBONE
    cfg.backbone.pretrained = True
    cfg.backbone.frozen     = frozen
    cfg.backbone.img_size   = cfg.data.image_size  
    cfg.fusion.mode         = fusion_mode
    cfg.train.epochs        = 50
    cfg.experiment_name     = f"{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes               = f"CIFAKE · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

# Load data
cfg_data = make_cfg("joint_only") # backbone settings only, fusion mode ignored
train_loader, val_loader, test_loader = get_cifake_loaders(cfg_data)
print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}")

Train: 85,000  Val: 15,000  Test: 20,000
Train: 85,000  Val: 15,000  Test: 20,000


## Experiment 0: Spatial-Only Baseline
Trains only the spatial backbone as a standalone classifier with no frequency branch.
This gives the honest floor. How well the backbone alone can classify real vs fake.
All delta values in later experiments are relative to this number.


In [ ]:
cfg0 = make_cfg("joint_only")  # backbone settings only, fusion mode ignored
cfg0.experiment_name = f"{BACKBONE}_spatial_only"
cfg0.notes           = f"spatial-only baseline, no freq branch, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")
print("All subsequent delta values are relative to this number.")

Device: cuda


Training spatial-only baseline (convnext_base) for 50 epochs...
Train: 85,000  Val: 15,000


Epoch   1/50 | train_loss=0.1203 | val_acc=97.0%


Epoch   2/50 | train_loss=0.0678 | val_acc=98.1%


Epoch   3/50 | train_loss=0.0505 | val_acc=98.2%


...

Epoch  36/50 | train_loss=0.0011 | val_acc=98.6%


Epoch  37/50 | train_loss=0.0014 | val_acc=98.7%


Epoch  38/50 | train_loss=0.0011 | val_acc=98.8%


Epoch  39/50 | train_loss=0.0012 | val_acc=98.8%


Epoch  40/50 | train_loss=0.0011 | val_acc=98.8%


## Experiment 1: Joint-Only Baseline
Both branches concatenated into a single shared classifier. No weighting between branches.
This is the floor — all other experiments are compared against this delta value.


In [7]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)

Running: convnext_base_joint_only
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.3629 | val_acc=97.6% | val_auc=0.997 | val_f1=0.976
  grad norms — freq=0.1653 | spatial=0.9810
  -> Saved best val_acc=97.6%


Epoch   2/50 | train_loss=0.2553 | val_acc=97.7% | val_auc=0.998 | val_f1=0.978
  -> Saved best val_acc=97.7%


Epoch   3/50 | train_loss=0.2206 | val_acc=98.1% | val_auc=0.998 | val_f1=0.981
  -> Saved best val_acc=98.1%


Epoch   4/50 | train_loss=0.2016 | val_acc=97.7% | val_auc=0.998 | val_f1=0.977


...

Epoch  47/50 | train_loss=0.0777 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988


Epoch  48/50 | train_loss=0.0773 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988


Epoch  49/50 | train_loss=0.0776 | val_acc=98.8% | val_auc=0.998 | val_f1=0.988


Epoch  50/50 | train_loss=0.0771 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
Results saved → ./results/results.csv  (convnext_base_joint_only, acc=0.9873)

Training complete. Best val accuracy: 98.8%


0.9879333333333333

In [9]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_joint_only.pt

EVALUATION — convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
  Joint accuracy:     98.7%
  AUC-ROC:            0.999
  F1:                 0.987
  Spatial-only:       98.7%
  Freq-only:          92.3%
  Delta (Δ):          +0.0%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_joint_only, acc=0.9869)


## Experiment 2: Scalar Fusion
Two learned scalars (a, b) weight each branch. Softmax ensures a+b=1 always.
Watch the scalar values logged each epoch — b is the frequency weight.
If b drops below 0.1 early in training the frequency branch is being ignored.


In [10]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)

Running: convnext_base_scalar
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.3615 | val_acc=96.9% | val_auc=0.997 | val_f1=0.969
  scalars — spatial=0.498 freq=0.502
  grad norms — freq=0.4690 | spatial=0.8726
  -> Saved best val_acc=96.9%


Epoch   2/50 | train_loss=0.2569 | val_acc=98.1% | val_auc=0.998 | val_f1=0.981
  scalars — spatial=0.496 freq=0.504
  -> Saved best val_acc=98.1%


Epoch   3/50 | train_loss=0.2234 | val_acc=98.4% | val_auc=0.999 | val_f1=0.984
  scalars — spatial=0.494 freq=0.506
  -> Saved best val_acc=98.4%


Epoch   4/50 | train_loss=0.2017 | val_acc=98.3% | val_auc=0.999 | val_f1=0.983
  scalars — spatial=0.494 freq=0.506


...

Epoch  47/50 | train_loss=0.0857 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
  scalars — spatial=0.482 freq=0.518


Epoch  48/50 | train_loss=0.0852 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
  scalars — spatial=0.482 freq=0.518


Epoch  49/50 | train_loss=0.0871 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
  scalars — spatial=0.482 freq=0.518


Epoch  50/50 | train_loss=0.0854 | val_acc=98.7% | val_auc=0.998 | val_f1=0.987
  scalars — spatial=0.482 freq=0.518
Results saved → ./results/results.csv  (convnext_base_scalar, acc=0.9871)

Training complete. Best val accuracy: 98.8%


0.9875333333333334

In [11]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_scalar.pt

EVALUATION — convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
  Joint accuracy:     98.6%
  AUC-ROC:            0.998
  F1:                 0.986
  Spatial-only:       98.6%
  Freq-only:          91.3%
  Delta (Δ):          +0.1%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_scalar, acc=0.9864)


## Experiment 3: Gating Fusion 
A small MLP outputs a per-image gate value in [0,1] controlling how much to trust the frequency branch.
Key metric beyond accuracy: **gate entropy must be > 0.3 nats**.
Below that the gate is outputting near-constant values and is not genuinely adapting per sample.


In [12]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)

Running: convnext_base_gating
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: convnext_base_gating
Backbone: convnext_base | Fusion: gating | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.2050 | val_acc=97.4% | val_auc=0.998 | val_f1=0.974
  gate — entropy=1.661 nats | mean=0.415 | var=0.0040
  grad norms — freq=0.9902 | spatial=0.1378
  -> Saved best val_acc=97.4%


Epoch   2/50 | train_loss=0.0957 | val_acc=97.2% | val_auc=0.999 | val_f1=0.972
  gate — entropy=1.163 nats | mean=0.553 | var=0.0021


Epoch   3/50 | train_loss=0.0683 | val_acc=98.2% | val_auc=0.999 | val_f1=0.982
  gate — entropy=1.314 nats | mean=0.520 | var=0.0021
  -> Saved best val_acc=98.2%


Epoch   4/50 | train_loss=0.0564 | val_acc=98.1% | val_auc=0.998 | val_f1=0.981
  gate — entropy=1.138 nats | mean=0.574 | var=0.0013


...

Epoch  47/50 | train_loss=0.0053 | val_acc=98.8% | val_auc=0.999 | val_f1=0.988
  gate — entropy=0.746 nats | mean=0.343 | var=0.0011


Epoch  48/50 | train_loss=0.0045 | val_acc=98.8% | val_auc=0.999 | val_f1=0.988
  gate — entropy=0.645 nats | mean=0.334 | var=0.0012


Epoch  49/50 | train_loss=-0.0010 | val_acc=98.8% | val_auc=0.999 | val_f1=0.988
  gate — entropy=0.935 nats | mean=0.336 | var=0.0014


Epoch  50/50 | train_loss=-0.0099 | val_acc=98.8% | val_auc=0.999 | val_f1=0.989
  gate — entropy=1.031 nats | mean=0.332 | var=0.0014
Results saved → ./results/results.csv  (convnext_base_gating, acc=0.9885)

Training complete. Best val accuracy: 98.9%


0.9887333333333334

In [13]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_gating.pt

EVALUATION — convnext_base_gating
Backbone: convnext_base | Fusion: gating | Frozen: False
  Joint accuracy:     98.7%
  AUC-ROC:            0.999
  F1:                 0.987
  Spatial-only:       98.6%
  Freq-only:          93.2%
  Delta (Δ):          +0.1%  (freq branch contribution)

  Gate distribution:
    entropy:  0.917 nats  (OK)
    mean:     0.361
    variance: 0.0010

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_gating, acc=0.987)


## Experiment 4: Gating Fusion, Frozen Backbone
Same as Experiment 3 but backbone weights are locked.
Only the projection head, frequency branch, fusion, and joint head train.

If the frequency branch helps more here than in Experiment 3, it means the backbone
was learning to capture spectral information during fine-tuning in Experiment 3 —
essentially teaching itself what the frequency branch provides.


In [3]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)

Running: convnext_base_gating_frozen
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell



Experiment: convnext_base_gating_frozen
Backbone: convnext_base | Fusion: gating | Frozen: True
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch 1:   0%|          | 0/1329 [00:00<?, ?batch/s]/home/peter-kabati/Desktop/DeepLearning(overall)/asfr/utils/fft_utils.py:37: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at /pytorch/aten/src/ATen/EmptyTensor.cpp:54.)
  spectrum = torch.fft.fft2(image_tensor)


Epoch   1/50 | train_loss=0.5305 | val_acc=93.3% | val_auc=0.982 | val_f1=0.935
  gate — entropy=1.936 nats | mean=0.506 | var=0.0071
  grad norms — freq=0.9122 | spatial=0.3843
  -> Saved best val_acc=93.3%


Epoch   2/50 | train_loss=0.4236 | val_acc=92.3% | val_auc=0.986 | val_f1=0.928
  gate — entropy=2.184 nats | mean=0.580 | var=0.0116


...

Epoch  47/50 | train_loss=0.2096 | val_acc=96.6% | val_auc=0.995 | val_f1=0.967
  gate — entropy=2.746 nats | mean=0.583 | var=0.0416


Epoch  48/50 | train_loss=0.2095 | val_acc=96.6% | val_auc=0.995 | val_f1=0.966
  gate — entropy=2.751 nats | mean=0.580 | var=0.0419


Epoch  49/50 | train_loss=0.2078 | val_acc=96.5% | val_auc=0.995 | val_f1=0.966
  gate — entropy=2.755 nats | mean=0.576 | var=0.0422


Epoch  50/50 | train_loss=0.2084 | val_acc=96.6% | val_auc=0.995 | val_f1=0.966
  gate — entropy=2.748 nats | mean=0.578 | var=0.0416

Training complete. Best val accuracy: 96.7%
Results will be logged to CSV after full_evaluation().


0.967

In [4]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_convnext_base_gating_frozen.pt

EVALUATION — convnext_base_gating_frozen
Backbone: convnext_base | Fusion: gating | Frozen: True
  Joint accuracy:     96.5%
  AUC-ROC:            0.995
  F1:                 0.966
  Spatial-only:       90.7%
  Freq-only:          92.8%
  Delta (Δ):          +5.8%  (freq branch contribution)

  Gate distribution:
    entropy:  2.750 nats  (OK)
    mean:     0.583
    variance: 0.0413

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (convnext_base_gating_frozen, acc=0.965)


## Summary: convnext_base
Compare all four runs. Key columns: accuracy, delta (freq branch contribution), gate_entropy.


In [6]:
df = pd.read_csv("./results/results.csv")
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))

            experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
         freq_only_baseline     gating   False    0.9443   0.9874 0.9450        0.0000
 convnext_base_spatial_only joint_only   False    0.9892   0.9981 0.9893        0.0000
   convnext_base_joint_only joint_only   False    0.9869   0.9988 0.9869        0.0000
       convnext_base_scalar     scalar   False    0.9864   0.9983 0.9864        0.0000
       convnext_base_gating     gating   False    0.9870   0.9988 0.9870        0.9172
convnext_base_gating_frozen     gating    True    0.9650   0.9953 0.9655        2.7496


**What to look for:**
- **Delta** (printed by full_evaluation) - how much the frequency branch added over spatial alone
- **Gate entropy** - must be > 0.3 nats for gating experiments to be valid
- **Frozen vs fine-tuned** - if frozen delta > fine-tuned delta, the backbone was absorbing spectral information during fine-tuning
